In [ ]:
import pandas as pd
import numpy as np

import optuna

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import StackingClassifier
from xgboost import XGBClassifier

from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

import joblib

### 1. Data Loading & Cleaning

In [ ]:
def load_fasttext_data(filepath):
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split(None, 1)
            if len(parts) == 2:
                label, text = parts
                label = label.replace('__label__', '')
                data.append([label, text])
    df = pd.DataFrame(data, columns=['label', 'text'])
    df['label'] = pd.to_numeric(df['label'], errors='coerce')
    return df.dropna(subset=['label'])

df1 = load_fasttext_data('Datasets/dataset 1.txt')
df2 = load_fasttext_data('Datasets/dataset 2.txt')
df3 = load_fasttext_data('Datasets/dataset 3.txt')

df4 = pd.read_csv('Datasets/dataset 4.csv')[['class_index', 'review_text']] \
        .rename(columns={'class_index': 'label', 'review_text': 'text'})

df5 = pd.read_csv('Datasets/Neutral Dataset.csv')[['class_index', 'review_text']] \
        .rename(columns={'class_index': 'label', 'review_text': 'text'})

df4['label'] = pd.to_numeric(df4['label'], errors='coerce')
df5['label'] = pd.to_numeric(df5['label'], errors='coerce')

df4 = df4.dropna(subset=['label'])
df5 = df5.dropna(subset=['label'])

combined_dataset = pd.concat([df1, df2, df3, df4, df5], ignore_index=True)

combined_dataset['label'] = combined_dataset['label'].astype(int)
combined_dataset = combined_dataset.dropna(subset=['text'])

label_mapping = {
    1: "Negative",
    2: "Neutral",
    3: "Positive"
}

combined_dataset['category'] = combined_dataset['label'].map(label_mapping)
combined_dataset = combined_dataset.dropna(subset=['category'])
combined_dataset = combined_dataset.sample(frac=1, random_state=42).reset_index(drop=True)

X = combined_dataset['text']
y = combined_dataset['category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

custom_stop_words = list(
    ENGLISH_STOP_WORDS.difference({'not', 'no', 'very', 'but', 'against'})
)

print("Dataset shape:", combined_dataset.shape)
print("Class distribution:\n", combined_dataset['category'].value_counts())

Dataset shape: (791855, 3)
Class distribution:
 category
Positive    264963
Negative    264650
Neutral     262242
Name: count, dtype: int64


### 2. Optuna Optimization

In [4]:
sample_idx = X_train.sample(n=80000, random_state=42).index
X_sample = X_train.loc[sample_idx]
y_sample = y_train.loc[sample_idx]

def objective(trial):
    classifier_name = trial.suggest_categorical(
        'classifier', ['LogisticRegression', 'LinearSVC', 'MultinomialNB']
    )
    
    if classifier_name == 'LogisticRegression':
        clf = LogisticRegression(
            C=trial.suggest_float('lr_c', 0.1, 5.0, log=True),
            solver='saga',
            class_weight='balanced',
            max_iter=2000
        )
        
    elif classifier_name == 'LinearSVC':
        clf = LinearSVC(
            C=trial.suggest_float('svc_c', 0.01, 1.0, log=True),
            class_weight='balanced',
            max_iter=2000
        )
    
    elif classifier_name == 'MultinomialNB':
        clf = MultinomialNB(
            alpha=trial.suggest_float('nb_alpha', 0.01, 10.0, log=True)
        )

    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(
            stop_words=custom_stop_words,
            max_features=120000,
            ngram_range=(1, 3),
            sublinear_tf=True
        )),
        ('clf', clf)
    ])

    return cross_val_score(pipeline, X_sample, y_sample, n_jobs=3, cv=3).mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=40)


[I 2026-04-08 20:41:07,286] A new study created in memory with name: no-name-6ffce180-58ba-497b-81c3-e2a716abe68d
[I 2026-04-08 20:41:31,699] Trial 0 finished with value: 0.6302000384786685 and parameters: {'classifier': 'MultinomialNB', 'nb_alpha': 1.4489009956106838}. Best is trial 0 with value: 0.6302000384786685.
[I 2026-04-08 20:41:54,099] Trial 1 finished with value: 0.6490375219218483 and parameters: {'classifier': 'LinearSVC', 'svc_c': 0.08703248951715496}. Best is trial 1 with value: 0.6490375219218483.
[I 2026-04-08 20:42:15,296] Trial 2 finished with value: 0.5985875304999398 and parameters: {'classifier': 'MultinomialNB', 'nb_alpha': 0.03828118370231433}. Best is trial 1 with value: 0.6490375219218483.
[I 2026-04-08 20:42:38,142] Trial 3 finished with value: 0.6497375225470748 and parameters: {'classifier': 'LogisticRegression', 'lr_c': 0.5272177385686663}. Best is trial 3 with value: 0.6497375225470748.
[I 2026-04-08 20:42:58,339] Trial 4 finished with value: 0.59086252721

### 3. Final Stacking Ensemble

In [5]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)

bp = study.best_params
print(f"\nBest Params Found: {bp}")

estimators = [
    ('lr', LogisticRegression(
        C=bp.get('lr_c', 0.5),
        solver='saga',
        class_weight='balanced',
        max_iter=1000
    )),
    
    ('svc', LinearSVC(
        C=bp.get('svc_c', 0.1),
        class_weight='balanced',
        max_iter=3000
    )),

    ('nb', MultinomialNB(
        alpha=bp.get('nb_alpha', 1.0)
    ))
]

stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=XGBClassifier(
        use_label_encoder=False,
        eval_metric='mlogloss',
        n_jobs=1
    ),cv=3, n_jobs=3
)

final_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words=custom_stop_words,
        max_features=100000,
        ngram_range=(1, 3),
        sublinear_tf=True
    )),
    ('stacker', stacking_clf)
])

final_pipeline.fit(X_train, y_train)



Best Params Found: {'classifier': 'LogisticRegression', 'lr_c': 0.6774478334887627}


c:\Users\sugam\Documents\EduSync\backend\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [21:03:59] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidf', ...), ('stacker', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


### 4. Final Training & Prediction

In [6]:
y_pred = final_pipeline.predict(X_test)
print("\n" + "="*50)
print("FINAL STACKING PERFORMANCE (3 CLASSES - LR+SVC+NB ENSEMBLE)")
print("="*50)
print(classification_report(y_test, y_pred))



FINAL STACKING PERFORMANCE (3 CLASSES - LR+SVC+NB ENSEMBLE)
              precision    recall  f1-score   support

    Negative       0.72      0.73      0.72     52930
     Neutral       0.58      0.57      0.58     52448
    Positive       0.75      0.76      0.75     52993

    accuracy                           0.69    158371
   macro avg       0.68      0.69      0.68    158371
weighted avg       0.68      0.69      0.69    158371



In [15]:
def predict_sentiment(text):
    return final_pipeline.predict([text])[0]

test_sentence = "The student is doing great in studies but the behaviour is not good in class."
print(f'\nTest sentence: "{test_sentence}"')
print(f'Predicted category: {predict_sentiment(test_sentence)}')


Test sentence: "The student is doing great in studies but the behaviour is not good in class."
Predicted category: Neutral


### 5. Exporting model

In [14]:
model_filename = 'sentiment_analysis_model.pkl'
joblib.dump(final_pipeline, model_filename)

['sentiment_analysis_model.pkl']

### 6. Testing model

In [1]:
from joblib import load

model = load("sentiment_analysis_model.pkl")

sentence = ["What is this shit."]

print(model.predict(sentence))

['Neutral']


C:\Users\Sugam Shrestha\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\pickle.py:1754: UserWarning: [03:19:48] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\gbm\../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)
